# CRISP-DM Phase 1 – Business Understanding

**Project:** Arbovirus differential diagnosis with explainable AI — a "smart hospital" module for the first consultation
of a febrile patient.

The project follows **CRISP-DM**. Each phase has a notebook and a page in the web app:

| # | CRISP-DM phase | Notebook | Web page |
|---|---|---|---|
| 1 | Business Understanding | `01_business_understanding.ipynb` | `/methodology/1` |
| 2 | Data Understanding | `02_data_understanding.ipynb` | `/methodology/2` |
| 3 | Data Preparation | `03_data_preparation.ipynb` | `/methodology/3` |
| 4 | Modeling | `04_modeling.ipynb` | `/methodology/4` |
| 5 | Evaluation | `05_evaluation.ipynb` | `/methodology/5` |
| 6 | Deployment | `06_deployment.ipynb` | `/methodology/6` |

The process is **iterative**: results of later phases send the work back to earlier ones.

## 1.1 Problem

Dengue, chikungunya and Zika are transmitted by the same mosquito (*Aedes aegypti*), co-circulate in Colombia and Brazil
and begin with almost the same picture: fever, headache, muscle and joint pain, rash. At the first consultation the clinician
must decide **what to suspect, which test to order and how to follow the patient**. Errors matter:

- Dengue can progress to **severe dengue**; its warning signs must be watched.
- The anti-inflammatory drugs (NSAIDs) used for chikungunya's joint pain **are dangerous in dengue**.

## 1.2 How the objective evolved (CRISP-DM is a cycle)

| Iteration | Objective | Outcome |
|---|---|---|
| 0 | Forecast weekly dengue cases | Kept as the baseline forecast on the home page |
| 1 | Binary screening: *dengue yes/no* (SIVIGILA Tuluá, Colombia) | Worked (ROC-AUC ≈ 0.63) but judged **too narrow** |
| **2 (current)** | **Differential diagnosis among arboviruses + explanations** | This version |

## 1.3 Business objective

Support the clinician at the first consultation of a febrile patient: which arbovirus to suspect, which test to order and
which warning signs to watch, **with a transparent explanation** of the suggestion.

> ⚠️ Clinical **decision support**, not a diagnosis. Confirmation is always clinical and by laboratory.

## 1.4 Data-mining objective

A **multiclass classifier**: `dengue` / `chikungunya` / `other_febrile`, from information available at the first
consultation — age, sex, days since onset, symptoms, basic clinical and laboratory findings, comorbidities and the
**epidemiological context** (state of residence and month: local circulation) — plus **SHAP explanations** for every
prediction. Target set by the course: macro ROC-AUC **above 0.80**.

Candidate algorithms (well suited to tabular clinical data): **Random Forest, XGBoost and LightGBM**, compared against a
linear baseline (Logistic Regression).

## 1.5 Success criteria (fixed before modeling)

| Criterion | Target | Why |
|---|---|---|
| Dengue recall | **≥ 80 %** | Missing dengue is the most dangerous error |
| Macro ROC-AUC (one-vs-rest) | **≥ 0.80** | Rank the three diagnoses well (target set by the course) |
| Better than the linear baseline | yes | Justifies the complexity of tree ensembles |
| Explainability | every prediction | A clinician must see *why* |
| Data leakage | none | Only information available at the first consultation |

## 1.6 Resources, constraints and risks

| Item | Detail |
|---|---|
| Data | SINAN (Brazil, DATASUS) for training; SIVIGILA Tuluá (Colombia) for external validation |
| Tools | Python, pandas, scikit-learn, XGBoost, LightGBM, SHAP, Flask, SQLAlchemy, Render |
| Constraint | No public dataset has **numeric** lab values (platelet or white-cell counts) |
| Constraint | Zika's public notification form has **no clinical fields** |
| Risk | Overlapping symptoms limit how well any model can separate the diseases |
| Risk | A model trained in Brazil may not transfer to Colombia → external validation |
| Ethics | Decision support only, visible notice, transparent explanations |

## 1.7 Project plan

| Phase | Main output | Code |
|---|---|---|
| 2. Data Understanding | Source inventory, label quality, clinical signal | `src/dbc.py`, `src/sinan.py` |
| 3. Data Preparation | Lab-confirmed, class-balanced datasets 2023 and 2025 | `src/sinan.py`, `src/preprocessing.py` |
| 4. Modeling | Four models compared, dengue alert threshold | `src/train.py` |
| 5. Evaluation | Test, SHAP, temporal (2025) and external (Colombia) validation | `src/train.py`, `src/explain.py` |
| 6. Deployment | Web app with per-patient explanations | `app.py`, `src/prediction.py` |